In [3]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "ready_for_ML.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError("Could not locate project root containing ready_for_ML.py")

from ready_for_ML import FootballPreprocessor
from xgboost import XGBClassifier
import pandas as pd
from sklearn.metrics import classification_report


In [5]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")

# --- Chronological season split ---
train_seasons = ["21-22", "22-23", "23-24", "24-25"]
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)].copy()
test_df = all_df[all_df["Season"] == test_season].copy()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

X_train = train_df.drop(columns=drop_cols)
y_train_raw = train_df["FTR"]

X_test = test_df.drop(columns=drop_cols)
y_test_raw = test_df["FTR"]

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# --- Encode FTR (A/D/H) into 0/1/2 ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# --- Train ---
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=10,
    min_child_weight=5,   # XGBoost's equivalent of min_samples_leaf
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss"
)
xgb.fit(X_train, y_train)

# --- Predict & evaluate ---
y_pred = xgb.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix (rows=actual, cols=predicted, order = classes below):")
print(le.classes_)
print(confusion_matrix(y_test, y_pred))

Train shape: (1439, 49)
Test shape: (360, 49)
Class mapping: {'A': np.int64(0), 'D': np.int64(1), 'H': np.int64(2)}

Classification Report:
              precision    recall  f1-score   support

           A       0.40      0.41      0.41       109
           D       0.38      0.08      0.13        99
           H       0.48      0.72      0.58       152

    accuracy                           0.45       360
   macro avg       0.42      0.41      0.37       360
weighted avg       0.43      0.45      0.41       360

Confusion Matrix (rows=actual, cols=predicted, order = classes below):
['A' 'D' 'H']
[[ 45   7  57]
 [ 31   8  60]
 [ 36   6 110]]


In [7]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta):
        unique_seasons = sorted(meta["Season"].unique())

        for i in range(1, len(unique_seasons)):
            train_seasons = unique_seasons[:i]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, test_idx, train_seasons, test_season)


# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")
all_df = all_df.reset_index(drop=True)  # important: validator yields positional-style index, keep it clean

print(f"Full dataset shape: {all_df.shape}")
print(f"Seasons found: {sorted(all_df['Season'].unique())}")

drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_results = []
all_importances = []
class_labels = sorted(all_df["FTR"].unique())  # fixed label order across folds for aggregation
label_encoder = LabelEncoder()
label_encoder.fit(class_labels)


for fold_num, (train_idx, test_idx, train_seasons, test_season) in enumerate(validator.split(all_df), start=1):

    train_df = all_df.loc[train_idx]
    test_df = all_df.loc[test_idx]

    print(f"\n{'='*60}")
    print(f"Fold {fold_num}: train on {train_seasons} -> test on {test_season}")
    print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")

    X_train = train_df.drop(columns=drop_cols)
    y_train = label_encoder.transform(train_df["FTR"])

    X_test = test_df.drop(columns=drop_cols)
    y_test = label_encoder.transform(test_df["FTR"])

    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    xgb = XGBClassifier(
    n_estimators=300,
    max_depth=10,
    min_child_weight=5,   # XGBoost's equivalent of min_samples_leaf
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss"
)
    xgb.fit(X_train, y_train)

    y_pred = xgb.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    print(f"Accuracy: {acc:.4f} | Macro F1: {f1_macro:.4f}")
    print(classification_report(y_test, y_pred, labels=label_encoder.transform(class_labels), target_names=class_labels))
    print("Confusion Matrix (rows=actual, cols=predicted):")
    print(class_labels)
    print(confusion_matrix(y_test, y_pred, labels=label_encoder.transform(class_labels)))

    fold_results.append({
        "fold": fold_num,
        "test_season": test_season,
        "train_seasons": train_seasons,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "n_train": len(train_df),
        "n_test": len(test_df),
    })

    all_importances.append(pd.Series(xgb.feature_importances_, index=X_train.columns))

# --- Aggregate across folds ---
results_df = pd.DataFrame(fold_results)
print(f"\n{'='*60}")
print("WALK-FORWARD SUMMARY")
print(f"{'='*60}")
print(results_df[["fold", "test_season", "n_train", "n_test", "accuracy", "f1_macro"]])

print(f"\nMean accuracy across folds: {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")
print(f"Mean macro F1 across folds: {results_df['f1_macro'].mean():.4f} (+/- {results_df['f1_macro'].std():.4f})")

# --- Aggregate feature importance across folds ---
importance_df = pd.concat(all_importances, axis=1)
importance_df.columns = [f"fold_{r['fold']}_{r['test_season']}" for r in fold_results]
mean_importance = importance_df.mean(axis=1).sort_values(ascending=False)

print("\nTop 15 features by mean importance across all folds:")
print(mean_importance.head(15))

Full dataset shape: (5393, 49)
Seasons found: ['11-12', '12-13', '13-14', '14-15', '15-16', '16-17', '17-18', '18-19', '19-20', '20-21', '21-22', '22-23', '23-24', '24-25', '25-26']

Fold 1: train on ['11-12'] -> test on 12-13
Train shape: (359, 49) | Test shape: (359, 49)
Accuracy: 0.4457 | Macro F1: 0.3990
              precision    recall  f1-score   support

           A       0.37      0.33      0.35       102
           D       0.31      0.22      0.26       101
           H       0.53      0.67      0.59       156

    accuracy                           0.45       359
   macro avg       0.40      0.41      0.40       359
weighted avg       0.42      0.45      0.43       359

Confusion Matrix (rows=actual, cols=predicted):
['A', 'D', 'H']
[[ 34  27  41]
 [ 27  22  52]
 [ 31  21 104]]

Fold 2: train on ['11-12', '12-13'] -> test on 13-14
Train shape: (718, 49) | Test shape: (360, 49)
Accuracy: 0.4917 | Macro F1: 0.4501
              precision    recall  f1-score   support

       

In [2]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- Same season-split logic as the single-run cell above, but regularized to
#     fix the overfitting we found (that config hit 100% train accuracy) ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_with_bookies.csv")

train_seasons = ["21-22", "22-23", "23-24"]
val_season = "24-25"   # held out from training, used only for early stopping
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)].copy()
val_df = all_df[all_df["Season"] == val_season].copy()
test_df = all_df[all_df["Season"] == test_season].copy()

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

drop_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season",
    "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds", "NumBookies", "B365HomeOdds", "B365DrawOdds", "B365AwayOdds"
]
X_train = train_df.drop(columns=drop_cols).fillna(0)
y_train_raw = train_df["FTR"]

X_val = val_df.drop(columns=drop_cols).fillna(0)
y_val_raw = val_df["FTR"]

X_test = test_df.drop(columns=drop_cols).fillna(0)
y_test_raw = test_df["FTR"]

# --- Encode FTR (A/D/H) into 0/1/2 ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val = le.transform(y_val_raw)
y_test = le.transform(y_test_raw)
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# --- Train, with the regularization the unconstrained version was missing ---
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,             # was 10 -- way too deep for ~1400 training rows
    learning_rate=0.05,      # was left at the default (~0.3)
    subsample=0.8,           # row subsampling per tree, like RF's bootstrap
    colsample_bytree=0.8,    # feature subsampling per tree, like RF's random subset
    min_child_weight=5,
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
    early_stopping_rounds=20,
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f"\nBest iteration: {xgb.best_iteration} (out of {xgb.n_estimators} max) -- early stopping cut off the rest")

# --- Predict & evaluate ---
y_pred = xgb.predict(X_test)

print(f"\nTrain accuracy: {accuracy_score(y_train, xgb.predict(X_train)):.4f} ")
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix (rows=actual, cols=predicted, order = classes below):")
print(le.classes_)
print(confusion_matrix(y_test, y_pred))

Train shape: (1079, 106)
Val shape: (360, 106)
Test shape: (360, 106)
Class mapping: {'A': np.int64(0), 'D': np.int64(1), 'H': np.int64(2)}

Best iteration: 28 (out of 300 max) -- early stopping cut off the rest

Train accuracy: 0.6756 
Test accuracy: 0.4806

Classification Report:
              precision    recall  f1-score   support

           A       0.42      0.39      0.41       109
           D       0.00      0.00      0.00        99
           H       0.50      0.86      0.63       152

    accuracy                           0.48       360
   macro avg       0.31      0.42      0.35       360
weighted avg       0.34      0.48      0.39       360

Confusion Matrix (rows=actual, cols=predicted, order = classes below):
['A' 'D' 'H']
[[ 43   0  66]
 [ 37   0  62]
 [ 22   0 130]]


c:\Users\misog\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\misog\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\misog\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [15]:
#SAME STRUCTURE AS THE LAST RF CELL IN regression_models.ipynb (window=3, xG features,
#compared vs BET365), swapped to XGBoost. XGBoost needs integer labels (not 'A'/'D'/'H'
#strings), so a LabelEncoder is fit once up front and reused across every fold for
#consistent encoding, then mapped back to le.classes_ order when building fair_market_proba.
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import label_binarize, LabelEncoder
from scipy import stats

# --- Walk-forward across every available season, with a validation season
#     carved out of each training window so overfitting is visible fold-by-fold
#     (train log loss vs. validation log loss), not just inferred after the fact.
#
#     WINDOW and VAL_SEASONS are the ONLY place to change these -- they're passed
#     through as keyword args below, not re-hardcoded at the call site, to avoid
#     the earlier bug where the default said one thing and the call site silently
#     used another. ---

WINDOW = 4       # total seasons of history pulled into each fold
VAL_SEASONS = 1   # how many of the most-recent seasons in that window are held out for validation

all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons_with_bookies.csv")
all_df = all_df.reset_index(drop=True)
all_df = all_df[["Home_Elo", "Away_Elo", "Home_xG_Rolling5", "Away_xGA_Rolling5", "Home_TablePosDiff", "Away_Elo", "Home_Elo", "Away_xG_Rolling5", "Home_xGA_Rolling5", "Away_TablePosDiff", "FTR", "Season"]]

drop_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season",
    "AvgHomeOdds", "AvgDrawOdds", "AvgAwayOdds", "NumBookies", "B365HomeOdds", "B365DrawOdds", "B365AwayOdds"
]

le = LabelEncoder()
le.fit(all_df["FTR"])  # fit once on the full label set so encoding is consistent across every fold


class WalkForwardValidator:

    def __init__(self, seasons):
        self.seasons = seasons

    def split(self, meta, window, val_seasons=1):
        unique_seasons = sorted(meta["Season"].unique())
        for i in range(window, len(unique_seasons)):
            window_seasons = unique_seasons[i - window:i]
            train_seasons = window_seasons[:-val_seasons]
            val_season_list = window_seasons[-val_seasons:]
            test_season = unique_seasons[i]

            train_idx = meta[meta["Season"].isin(train_seasons)].index
            val_idx = meta[meta["Season"].isin(val_season_list)].index
            test_idx = meta[meta["Season"] == test_season].index

            yield (train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season)


def eval_probs(model, X, y_encoded, n_classes):
    """Log loss + Brier score for a fitted model against a labeled set (y already integer-encoded)."""
    proba = model.predict_proba(X)
    y_onehot = label_binarize(y_encoded, classes=range(n_classes))
    eps = 1e-15
    ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1)).mean()
    brier = ((proba - y_onehot) ** 2).sum(axis=1).mean()
    return ll, brier


validator = WalkForwardValidator(seasons=sorted(all_df["Season"].unique()))

fold_rows = []
all_model_ll, all_BET365_ll = [], []
all_model_brier, all_BET365_brier = [], []

for train_idx, val_idx, test_idx, train_seasons, val_season_list, test_season in validator.split(
    all_df, window=WINDOW, val_seasons=VAL_SEASONS
):

    train_df = all_df.loc[train_idx]
    val_df = all_df.loc[val_idx]
    test_df = all_df.loc[test_idx]

    X_train = train_df.drop(columns=drop_cols).fillna(0)
    y_train = le.transform(train_df["FTR"])
    X_val = val_df.drop(columns=drop_cols).fillna(0)
    y_val = le.transform(val_df["FTR"])
    X_test = test_df.drop(columns=drop_cols).fillna(0)

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=2,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.764,
        min_child_weight=1,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss",
        reg_alpha=0.5,  # L1 regularization
        reg_lambda=0.13,  # L2 regularization

    )
    xgb.fit(X_train, y_train)
    classes_order = le.classes_  # ['A', 'D', 'H'], alphabetical -- matches le.transform's integer order

    # --- Train / validation log loss & Brier -- the overfitting check ---
    train_ll, train_brier = eval_probs(xgb, X_train, y_train, len(classes_order))
    val_ll, val_brier = eval_probs(xgb, X_val, y_val, len(classes_order))

    # --- Test set: model vs. de-vigged Bet365, same as the RF reference cell ---
    proba = xgb.predict_proba(X_test)
    overround = 1 / test_df["B365HomeOdds"] + 1 / test_df["B365DrawOdds"] + 1 / test_df["B365AwayOdds"]
    fair = {
        "H": (1 / test_df["B365HomeOdds"]) / overround,
        "D": (1 / test_df["B365DrawOdds"]) / overround,
        "A": (1 / test_df["B365AwayOdds"]) / overround,
    }
    fair_market_proba = np.column_stack([fair[c].values for c in classes_order])

    y_true_encoded = le.transform(test_df["FTR"].values)
    y_onehot = label_binarize(y_true_encoded, classes=range(len(classes_order)))

    eps = 1e-15
    model_ll = -np.log(np.clip((proba * y_onehot).sum(axis=1), eps, 1))
    BET365_ll = -np.log(np.clip((fair_market_proba * y_onehot).sum(axis=1), eps, 1))
    model_brier = ((proba - y_onehot) ** 2).sum(axis=1)
    BET365_brier = ((fair_market_proba - y_onehot) ** 2).sum(axis=1)

    all_model_ll.append(model_ll)
    all_BET365_ll.append(BET365_ll)
    all_model_brier.append(model_brier)
    all_BET365_brier.append(BET365_brier)

    fold_rows.append({
        "test_season": test_season,
        "train_seasons": train_seasons,
        "val_season": val_season_list,
        "n_test": len(test_df),
        "train_log_loss": train_ll,
        "val_log_loss": val_ll,
        "test_log_loss": model_ll.mean(),
        "BET365_log_loss": BET365_ll.mean(),
        "train_brier": train_brier,
        "val_brier": val_brier,
        "test_brier": model_brier.mean(),
        "BET365_brier": BET365_brier.mean(),
    })

    print(
        f"{test_season} (train {train_seasons[0]}..{train_seasons[-1]}, val {val_season_list[0]}): "
        f"log loss train={train_ll:.4f} val={val_ll:.4f} test={model_ll.mean():.4f} BET365={BET365_ll.mean():.4f}"
    )

fold_df = pd.DataFrame(fold_rows)
print("\n=== Per-fold summary (overfitting check: train vs. val vs. test) ===")
print(fold_df[["test_season", "train_log_loss", "val_log_loss", "test_log_loss", "BET365_log_loss"]].round(4))

# --- Pool every match across every fold's TEST set for the model-vs-market test ---
model_ll_all = np.concatenate(all_model_ll)
BET365_ll_all = np.concatenate(all_BET365_ll)
model_brier_all = np.concatenate(all_model_brier)
BET365_brier_all = np.concatenate(all_BET365_brier)

print(f"\n=== Pooled across all {len(fold_df)} folds ({len(model_ll_all)} test matches) ===")
print(f"Model log loss:  {model_ll_all.mean():.4f}")
print(f"BET365 log loss: {BET365_ll_all.mean():.4f}")
print(f"Model Brier:     {model_brier_all.mean():.4f}")
print(f"BET365 Brier:    {BET365_brier_all.mean():.4f}")

print(f"\nMean train log loss:      {fold_df['train_log_loss'].mean():.4f}")
print(f"Mean validation log loss: {fold_df['val_log_loss'].mean():.4f}")
print(
    f"Gap (val - train): {fold_df['val_log_loss'].mean() - fold_df['train_log_loss'].mean():.4f}  "
    "(large gap = overfitting -- the model fits the training seasons far better than unseen ones)"
)

t_stat, p_value = stats.ttest_rel(model_ll_all, BET365_ll_all)
print(f"\nPaired t-test (log loss, model vs BET365): t={t_stat:.3f}, p={p_value:.6f}")

diff = model_ll_all - BET365_ll_all
rng = np.random.default_rng(42)
n = len(diff)
boot_means = np.array([diff[rng.integers(0, n, n)].mean() for _ in range(10000)])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"Bootstrap 95% CI for mean log-loss diff (model - BET365): [{ci_low:.4f}, {ci_high:.4f}]")

model_wins = (model_ll_all < BET365_ll_all).sum()
BET365_wins = (BET365_ll_all < model_ll_all).sum()
print(f"\nModel better on {model_wins}/{n} matches, BET365 better on {BET365_wins}/{n} matches")
print(f"Model better on {model_wins/n:.2%} of matches, BET365 better on {BET365_wins/n:.2%} of matches")

KeyError: "['Date' 'Time' 'HomeTeam' 'AwayTeam' 'FTHG' 'FTAG' 'FTR' 'Season'\n 'AvgHomeOdds' 'AvgDrawOdds' 'AvgAwayOdds' 'NumBookies' 'B365HomeOdds'\n 'B365DrawOdds' 'B365AwayOdds'] not found in axis"